In [0]:

# Load the data
import requests
import pandas as pd

def fetch_weather(lat, lon, timezone):
    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2013-01-01"
        "&end_date=2013-12-31"
        "&daily=temperature_2m_mean,precipitation_sum,windspeed_10m_max"
        f"&timezone={timezone}"
    )
    data = requests.get(url).json()

    return pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "temp": data["daily"]["temperature_2m_mean"],
        "precip": data["daily"]["precipitation_sum"],
        "wind": data["daily"]["windspeed_10m_max"]
    })


#ETCH DATA

chicago = fetch_weather(41.88, -87.63, "America/Chicago")
nyc = fetch_weather(40.71, -74.01, "America/New_York")

# SAVE AS TABLES

spark.createDataFrame(chicago).write.mode("overwrite").saveAsTable("chicago_weather")
spark.createDataFrame(nyc).write.mode("overwrite").saveAsTable("nyc_weather")

print("Tables created: chicago_weather, nyc_weather")

# QUICK EXPLORE
print("\nChicago summary:")
print(chicago.describe())
#SQL COMPARISON
spark.sql("""
SELECT
  c.month,
  ROUND(c.chicago_avg_temp, 2) AS chicago_avg_temp,
  ROUND(n.nyc_avg_temp, 2) AS nyc_avg_temp
FROM (
  SELECT MONTH(date) AS month, AVG(temp) AS chicago_avg_temp
  FROM chicago_weather
  GROUP BY MONTH(date)
) c
JOIN (
  SELECT MONTH(date) AS month, AVG(temp) AS nyc_avg_temp
  FROM nyc_weather
  GROUP BY MONTH(date)
) n
ON c.month = n.month
ORDER BY c.month
""").show()

Bring additional external Data (Air Quality Dataset)

In [0]:
import pandas as pd

url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/airquality.csv"
pdf = pd.read_csv(url)

pdf.shape
pdf.dtypes
pdf.head()

In [0]:
# Data cleaning
pdf = pdf.drop(columns=["Unnamed: 0"], errors="ignore")
pdf = pdf.dropna()

pdf.columns = [c.lower() for c in pdf.columns]

In [0]:
# Save Table 
df = spark.createDataFrame(pdf)
df.write.mode("overwrite").saveAsTable("airquality")

display(df)


In [0]:
%sql
SELECT *
FROM airquality
WHERE ozone > 100;



Aggregation

In [0]:
%sql
SELECT month, AVG(ozone) AS avg_ozone
FROM airquality
GROUP BY month
ORDER BY month;

Calculated Column

In [0]:
%sql
SELECT
  temp,
  wind,
  temp / wind AS temp_wind_ratio
FROM airquality;

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ozone vs Temp
sns.scatterplot(x='temp', y='ozone', data=pdf)
plt.title("Ozone vs Temperature")
plt.show()

Weather & Flight Delays Deep Dive

In [0]:
# Loading the data 
flights_df = spark.table("flights").toPandas()
weather_df = spark.table("nyc_weather").toPandas()

In [0]:
# Creating Daily delay
flights_df["date"] = pd.to_datetime(
    flights_df[["year","month","day"]]
)

daily_delay = flights_df.groupby("date")["arr_delay"].mean().reset_index()

In [0]:
# Merge weather data with daily delay
weather = pd.merge(daily_delay, weather_df, left_on="date", right_on="date")
# Drop missing values
weather = weather.dropna()
# Create a new column
weather["temp_wind_ratio"] = weather["temp"] / weather["wind"]
weather.head()

In [0]:
merged = daily_delay.merge(weather_df, on="date")

In [0]:
merged.corr(numeric_only=True)["arr_delay"]

Correlation with delays
Precipitation: 0.49 → strongest relationship
Wind: 0.19 → weak/moderate
Temperature: 0.10 → very weak
Main takeaway: rain matters most


In [0]:
# Correlation matrix
merged.corr()
# Plot correlation matrix
sns.heatmap(merged.corr(), annot=True)
plt.title("Correlation Matrix")
plt.show()

In [0]:
plt.scatter(merged["precip"], merged["arr_delay"])
plt.xlabel("Precipitation")
plt.ylabel("Avg Delay")
plt.title("Weather vs Flight Delays")
plt.show()

In [0]:
# Worst Delay Days
worst_days = merged.sort_values("arr_delay", ascending=False).head(5)
worst_days


- Pattern:
Most high-delay days include rain
One extreme case (June 13) has very heavy precipitation
Temperature varies → not a strong driver



Precipitation showed the strongest correlation with flight delays (0.49),
indicating that rain has a meaningful impact.